In [ ]:
# Copyright 2024 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

| | |
|-|-|
|Author(s) | [Lavi Nigam](https://github.com/lavinigam-gcp) |

Notebooks:
**goo.gle/io24-gemini-api**

Google AI Cookbook:
**goo.gle/google-ai-cookbook**

Vertex AI Cookbook:
**goo.gle/vertex-ai-cookbook**

# Step 1 - Google AI for Developers Gemini API

In [ ]:
# Library installation
! pip install PyPDF2
! apt-get install poppler-utils
! pip install pdf2image
! pip install --upgrade google-generativeai

### Restart current runtime

To use the newly installed packages in this Jupyter runtime, you must restart the runtime. You can do this by running the cell below, which will restart the current kernel.

In [ ]:
# Restart kernel after installs so that your environment can access the new packages
import IPython
import time

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

<div class="alert alert-block alert-warning">
<b>?? The kernel is going to restart. Please wait until it is finished before continuing to the next step. ??</b>
</div>



In [ ]:
# Downloading data from Google Drive
!gdown --folder https://drive.google.com/drive/folders/1RfMScB0Y1LUQdW5tvjyYA4_D21H5HMaT?usp=sharing -O /content/data/
print("Download completed")

In [ ]:
# Library
from IPython.display import display, Markdown, Latex
import google.generativeai as genai
import PyPDF2
from rich import print as rich_print
from rich.markdown import Markdown as rich_Markdown
import textwrap

In [ ]:
# Authentication
from google.colab import userdata
genai.configure(api_key=userdata.get('api_key'))

**[Gemini 1.5 Flash](https://ai.google.dev/gemini-api/docs/models/gemini#gemini-1.5-flash)**


![](https://storage.googleapis.com/github-repo/io24/gemini-api-session/img/model_1.png)

**RPM:** Requests per minute

**TPM:** Tokens per minute

**RPD:** Requests per day

**TPD:** Tokens per day

**[Gemini 1.5 Pro](https://ai.google.dev/gemini-api/docs/models/gemini#gemini-1.5-pro)**

![](https://storage.googleapis.com/github-repo/io24/gemini-api-session/img/model_2.png)

Gemini models are available in either preview or stable versions. In your code, you can use one of the following model name formats to specify which model and version you want to use.

* **Latest:** Points to the cutting-edge version of the model for a specified generation and variation. The underlying model is updated regularly and might be a preview version. Only exploratory testing apps and prototypes should use this alias.

    To specify the latest version, use the following pattern: `<model>-<generation>-<variation>-latest`. For example, `gemini-1.5-pro-latest`.

* **Latest stable:** Points to the most recent stable version released for the specified model generation and variation.

    To specify the latest stable version, use the following pattern: `<model>-<generation>-<variation>`. For example, `gemini-1.5-pro`.

* **Stable:** Points to a specific stable model. Stable models don't change. Most production apps should use a specific stable model.

    To specify a stable version, use the following pattern: `<model>-<generation>-<variation>-<version>`. For example, `gemini-1.5-pro-001`.

[More detail](https://ai.google.dev/gemini-api/docs/models/gemini#model-versions)


Gemini models are trained to work with the following languages: [Link](https://ai.google.dev/gemini-api/docs/models/gemini#available-languages)

In [ ]:
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

Here is the list of currently available models: [Link](https://ai.google.dev/gemini-api/docs/models/gemini#model-variations)

<div class="alert alert-block alert-warning">
<b>?? A token is equivalent to about 4 characters for Gemini models. 100 tokens are about 60-80 English words. ??</b>
</div>


In [ ]:
def to_markdown(text):
  text = text.replace('?', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

### API

#### Basic Usage

In [ ]:
# Set up the model
# Learn more about the API: https://ai.google.dev/gemini-api/docs/get-started/tutorial?lang=python
# More details about the models: https://ai.google.dev/gemini-api/docs/models/gemini

from google.generativeai.types import HarmCategory, HarmBlockThreshold, GenerationConfig

generation_config = GenerationConfig(
    temperature=1,
    max_output_tokens=8192,
)

# Explore more about safety settings: https://ai.google.dev/gemini-api/docs/safety-settings
# Explore Gemini's Safety guidance: https://ai.google.dev/gemini-api/docs/safety-guidance
safety_settings = {
    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
}

# works with text, code, images, video(with or without audio) and audio(mp3) with 2M input context - complex reasoning
model = genai.GenerativeModel(model_name="gemini-1.5-pro-001",  # using stable (not latest)
                              generation_config=generation_config,
                              safety_settings=safety_settings)

# works with text, code, images, video(with or without audio) and audio(mp3) with 1M input context - faster inference
model_flash = genai.GenerativeModel(model_name="gemini-1.5-flash-001", # using stable (not latest)
                              generation_config=generation_config,
                              safety_settings=safety_settings)

explore more examples of what's possible with Gemini: [Link](https://ai.google.dev/examples)

In [ ]:
%%time

response = model.generate_content("What is the meaning of life?")

In [ ]:
to_markdown(response.text)

In [ ]:
%%time

response = model_flash.generate_content("What is the meaning of life?")

In [ ]:
to_markdown(response.text)

In [ ]:
response.candidates

Explore API Troubleshooting and API errors here: [Link](https://ai.google.dev/gemini-api/docs/troubleshooting)

#### Streaming

In [ ]:
%%time
response = model_flash.generate_content("What is the meaning of life?", stream=True)

In [ ]:
for chunk in response:
  print(chunk.text)
  print("_"*80)

In [ ]:
response

#### Token Count

In [ ]:
model = genai.GenerativeModel('models/gemini-1.5-flash')
prompt = "What is life?"
response = model.generate_content(prompt)


print("count before response generation: ", model.count_tokens(prompt))
print("\nResponse: ",response.text)
print("\n Usage Metadata:\n",response.usage_metadata)

## Testing Hypothesis

### PDF (Extracted Text)

In [ ]:
#@title Helper Functions
## Helper Function

def pdf_to_dict_str(file_path):
    """Reads a PDF file and returns a dictionary with page numbers as keys and page text as values."""

    with open(file_path, 'rb') as pdf_file:
        pdf_reader = PyPDF2.PdfReader(pdf_file)
        num_pages = len(pdf_reader.pages)

        page_dict = {}
        for page_num in range(num_pages):
            page = pdf_reader.pages[page_num]
            page_text = page.extract_text()
            page_dict[page_num + 1] = page_text  # Page numbers start from 1

        return str(page_dict)

Source File: [Link](https://drive.google.com/file/d/1GXGfwoe8-vsoOM1E8SBOCybc7R5XRd2P/view?usp=drive_link) ~18 pages

In [ ]:
# read through each page and extracts the text and saves in the dictionary as {1:"context of page 1"}, 2:"context of page 2"}, .... }
pdf_context = pdf_to_dict_str('/content/data/Idea/stage_1_prototype_Google Cloud TPU blog.pdf')

In [ ]:
pdf_context[:100] #The whole dict is converted to strings so as to pass as context in the model

In [ ]:
rich_Markdown(eval(pdf_context)[1]) # Page 1 #use eval to convert string to dict object

In [ ]:
%%time

prompt_parts = [
  pdf_context,
  "What are key achievement for Google Cloud from the following blog. Mention in bullet.",
]

print(model.count_tokens(prompt_parts))
response = model.generate_content(prompt_parts)
print("\n Usage Metadata:\n",response.usage_metadata)
rich_Markdown(response.text)

In [ ]:
%%time

print(model_flash.count_tokens(prompt_parts))
response = model_flash.generate_content(prompt_parts)
print("\n Usage Metadata:\n",response.usage_metadata)
rich_Markdown(response.text)

Understand your billing: [Link](https://ai.google.dev/gemini-api/docs/billing)

Source File: [Link](https://drive.google.com/file/d/1iT0qe9EU8RQyiAlrMlJ-SH-E_WtMGFNm/view?usp=drive_link) ~15 pages

In [ ]:
%%time
prompt_parts = [
  pdf_to_dict_str('/content/data/Idea/goog-10-q-q2-2023-4-1-15.pdf'),
  """Answer based on the document provided:
- How has Google Cloud performed in last quarters?
- What factors have influenced its numebrs and is it positive or negative?
- What is the leadership view on Google Cloud business?
- What are some changes and decisions they have made along the way to achieve their goals?
- How are they doing with respect to other cloud companies?
  """,
]
print(model_flash.count_tokens(prompt_parts))
response = model.generate_content(prompt_parts)
print("\n Usage Metadata:\n",response.usage_metadata)
rich_Markdown(response.text)

### Audio File

Source File: [Link](https://drive.google.com/file/d/1kaq58S4oRmuZgDqK109uc6mwedVmPAby/view?usp=drive_link) ~1 hour

In [ ]:
%%time

earning_calls = genai.upload_file(path="/content/data/Idea/stage_1_prototype_Alphabet 2023 Q4 Earnings Call.mp3",
                            display_name="earning_calls")

In [ ]:
%%time

prompt_parts = [
  earning_calls,
  "what are major comments by Sundar and Ruth about AI and what key questions were asked by analyst and give their name? give answer in bullets ",
]
print(model.count_tokens(prompt_parts))
response = model.generate_content(prompt_parts,request_options={"timeout": 600})
print("\n Usage Metadata:\n",response.usage_metadata)
rich_Markdown(response.text)

In [ ]:
%%time

print(model_flash.count_tokens(prompt_parts))
response = model_flash.generate_content(prompt_parts)
print("\n Usage Metadata:\n",response.usage_metadata)
rich_Markdown(response.text)

#### Context Caching [Paid Feature]

More details about the API: [Link](https://ai.google.dev/gemini-api/docs/caching?lang=python)

Context Caching pricing: [Link](https://ai.google.dev/pricing)

In [ ]:
%%time

from google.generativeai import caching
import datetime
import time

# Create a cache with a 5 minute TTL
cache = caching.CachedContent.create(
    model='models/gemini-1.5-flash-001',
    display_name='alphabet earning call', # used to identify the cache
    system_instruction=(
        'You are an expert audio analyzer, and your job is to answer '
        'the user\'s query based on the audio file you have access to.'
    ),
    contents=[earning_calls],
    # caching duration is called the time to live (TTL). If not set, the TTL defaults to 1 hour.
    # The cost for caching depends on the input token size and how long you want the tokens to persist.
    ttl=datetime.timedelta(minutes=5),
)

In [ ]:
%%time

# Construct a GenerativeModel which uses the created cache.
cached_model = genai.GenerativeModel.from_cached_content(cached_content=cache)

# Query the model
response = cached_model.generate_content([("""Instructions:
- Quote the speaker name.
- Mention the timestamp.
- Respond in structured format and bullets.

Task: Summarize the questions by Bank of America analyst and their response from Alphabet.
""")])

print(response.usage_metadata)


rich_Markdown(response.text)

In [ ]:
# Query the model
response = cached_model.generate_content([("""Instructions:
- Quote the speaker name.
- Mention the timestamp.
- Respond in structured format and bullets.

Task: Summarize the questions by Morgan Stanley analyst and their response from Alphabet.
""")])

print(response.usage_metadata)


rich_Markdown(response.text)

In [ ]:
# delete the file
genai.delete_file(name=earning_calls.name)

### Images (Read PDF as Images)

In [ ]:
#@title Helper Functions
## Helper Function

import os
from pdf2image import convert_from_path

def pdf_to_images(file_path, output_folder="output_folder"):
    """Converts a PDF file into images for each page, saves them to a folder, and returns a list of image paths."""

    # Create output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Convert PDF to images
    images = convert_from_path(file_path, dpi=200, output_folder=output_folder, fmt="jpg")

    # Generate image paths
    image_paths = []
    pages_list = []
    upload_file_name = []
    for i, image in enumerate(images):
        image_name = f"page_{i+1}.jpg"  # Page numbers start from 1
        image_path = os.path.join(output_folder, image_name)
        image.save(image_path, "JPEG")
        image_paths.append(image_path)
        files_to_upload = genai.upload_file(image_path, display_name=image_path)
        pages_list.append(files_to_upload)
        upload_file_name.append([files_to_upload.name, files_to_upload.display_name])

    return image_paths, pages_list, upload_file_name

def make_request(prompt, files):
  request = [prompt]
  for file in files:
    request.append(file)
  return request

def delete_files(output_folder):
  for filename in os.listdir(output_folder):
    if filename.endswith(".jpg"):
      file_path = os.path.join(output_folder, filename)
      genai.delete_file(name=file_path)

Source File: [Link](https://drive.google.com/file/d/1GXGfwoe8-vsoOM1E8SBOCybc7R5XRd2P/view?usp=drive_link) ~18 pages

In [ ]:
%%time

file_path = "/content/data/Idea/stage_1_prototype_Google Cloud TPU blog.pdf"
output_folder = "/content/output_images"
image_paths, pages_list, upload_file_name = pdf_to_images(file_path, output_folder)

In [ ]:
image_paths[1]

In [ ]:
pages_list[1]

In [ ]:
upload_file_name[1]

In [ ]:
%%time

response = model_flash.generate_content(
make_request("what is the emfu for bf16 and 128b parameter model with 1 tpu v5e pod? Cite the table and page number and explain the significance of the results",
             pages_list))
print("\n Usage Metadata:\n",response.usage_metadata)
rich_Markdown(response.text)

In [ ]:
for filename in upload_file_name:
  print("Deleteing...",filename[0])
  genai.delete_file(name=filename[0])

### Video

Source File: [Link](https://drive.google.com/file/d/19aqpEZqTVTERbySYVrtDgSjuUBUabeSE/view?usp=drive_link)

In [ ]:
%%time
# Upload a video ~ 1min

video_file = genai.upload_file(path="/content/data/Idea/What's next for AI and Google Search _ Google I_O 2023.mp4",
                            display_name="product_pitch_google_video_file")

In [ ]:
%%time

prompt_parts = [
  video_file,
  "Describe this video and How is google using generative ai in search? Give response in bullet",
]
print(model.count_tokens(prompt_parts))
response = model.generate_content(prompt_parts,request_options={"timeout": 600})
print("\n Usage Metadata:\n",response.usage_metadata)
rich_Markdown(response.text)

You can also use cached context to cache the video and keep asking questions for larger video: [Link](https://ai.google.dev/gemini-api/docs/caching?lang=python#generate-content)

## Embeddings

In [ ]:
title = "The next generation of AI for developers and Google Workspace"
sample_text = ("Title: The next generation of AI for developers and Google Workspace"
    "\n"
    "Full article:\n"
    "\n"
    "Gemini API & Google AI Studio: An approachable way to explore and prototype with generative AI applications")

model = 'models/text-embedding-004'
embedding = genai.embed_content(model=model,
                                content=sample_text,
                                task_type="retrieval_document",
                                title=title)

print(embedding['embedding'][:5]) #printing only first 5

In [ ]:
len(embedding['embedding'])